<div style="background: linear-gradient(135deg, #000000 0%, #0a0a0a 50%, #1a0000 100%);
            padding: 30px 40px;
            border-radius: 4px;
            text-align: center;
            border-left: 4px solid #ff5347;
            border-right: 4px solid #ff5347;
            box-shadow: 0 0 40px rgba(255, 83, 71, 0.3), inset 0 0 60px rgba(255, 83, 71, 0.05);">
    <h1 style="color: #fafafa;
               font-family: monospace;
               font-size: 3.5em;
               letter-spacing: 0.5em;
               text-shadow: 0 0 20px rgba(255, 83, 71, 0.9), 0 0 40px rgba(255, 83, 71, 0.4);
               margin: 0;
               padding: 10px 0;">
        ML-KEM
    </h1>
    <div style="margin-top: 15px; color: #ff8a80; font-family: monospace; font-size: 0.75em; letter-spacing: 0.3em;">
        Michal Kowaslki
    </div>
</div>

In [1]:
import os

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.15.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs
import time
import numpy as np
import matplotlib.pyplot as plt

Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.15.0


c:\Users\Admin\Desktop\benchmark project\Benchmarking_Post-Quantum_Cryptographic_Algorithms\.venv\Lib\site-packages\oqs\__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (


In [2]:
import csv
import tracemalloc
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.serialization import Encoding, NoEncryption, PrivateFormat, PublicFormat

benchmark_output_path = "ml_kem_benchmark_results.txt"
csv_output_path = "ml_kem_timing.csv"
rsa_benchmark_output_path = "ecdh_p256_benchmark_results.txt"
rsa_csv_output_path = "ecdh_p256_timing.csv"


def write_header(file_path, header_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(header_text + "\n")
        f.write("=" * len(header_text) + "\n")

write_header(benchmark_output_path, "ML-KEM Benchmark Results")
write_header(rsa_benchmark_output_path, "ECDH-P-256 Benchmark Results")

csv_enabled = True
try:
    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
    with open(rsa_csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
except PermissionError:
    print(f"Warning: Cannot write to one or more CSV files (file may be open in another program). CSV logging disabled.")
    csv_enabled = False




def append_to_output(file_path, *lines):
    with open(file_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")


def append_to_csv(csv_path, metric, run, time_ms, peak_kb):
    if csv_enabled:
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms, peak_kb])

In [3]:
# Check available ML-KEM variants
print("Available ML-KEM algorithms:")
for alg in oqs.get_enabled_kem_mechanisms():
    if "ML-KEM" in alg:
        print(f"  - {alg}")

Available ML-KEM algorithms:
  - ML-KEM-512
  - ML-KEM-768
  - ML-KEM-1024


In [4]:
# Benchmark ML-KEM keypair generation for a representative parameter set
alg = "ML-KEM-768"
if alg not in oqs.get_enabled_kem_mechanisms():
    raise ValueError(f"Algorithm {alg} is not enabled. Choose one of: {[a for a in oqs.get_enabled_kem_mechanisms() if 'ML-KEM' in a]}")

print(f"Benchmarking {alg} keypair generation...")
print("Warming up...")
with oqs.KeyEncapsulation(alg) as kem:
    for _ in range(100):
        _ = kem.generate_keypair()
runs = 10000
times_ms = []
peaks_kb = []
with oqs.KeyEncapsulation(alg) as kem:
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        _ = kem.generate_keypair()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        times_ms.append(elapsed_ms)
        peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(times_ms):.3f} ms")
print(f"Min keypair time: {np.min(times_ms):.3f} ms")
print(f"Max keypair time: {np.mean(times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in times_ms])
print("Peaks (KB):", [round(p, 3) for p in peaks_kb])

append_to_output(
    benchmark_output_path,
    f"\nBenchmarking {alg} keypair generation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(times_ms):.3f} ms",
    f"Min keypair time: {np.min(times_ms):.3f} ms",
    f"Max keypair time: {np.max(times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(peaks_kb):.3f} KB")

Benchmarking ML-KEM-768 keypair generation...
Warming up...
Algorithm: ML-KEM-768
Runs: 10000
Mean keypair time: 0.401 ms
Min keypair time: 0.311 ms
Max keypair time: 0.401 ms
Mean peak memory: 4.923 KB
Max peak memory: 7.646 KB
Times (ms): [0.377, 0.4, 0.627, 0.444, 0.337, 0.329, 0.358, 0.322, 0.335, 0.402, 0.344, 0.334, 0.517, 0.53, 0.355, 0.52, 0.339, 0.461, 0.454, 0.422, 0.336, 0.335, 0.414, 0.367, 0.325, 0.341, 0.342, 0.325, 0.325, 0.353, 0.344, 0.381, 0.336, 0.327, 0.348, 0.475, 0.331, 0.339, 0.477, 0.339, 0.396, 0.346, 0.438, 0.559, 0.504, 0.475, 0.329, 0.37, 0.349, 0.324, 0.36, 0.319, 0.332, 0.372, 0.328, 0.325, 0.351, 0.384, 0.626, 0.367, 0.322, 0.374, 0.318, 0.355, 0.333, 0.324, 0.321, 0.313, 0.475, 0.433, 0.342, 0.326, 0.334, 0.32, 0.318, 0.378, 0.425, 1.013, 0.422, 0.441, 0.466, 0.379, 0.371, 0.426, 0.449, 0.358, 0.409, 0.341, 0.329, 0.323, 0.334, 0.413, 0.357, 0.521, 0.508, 0.384, 0.337, 0.323, 0.361, 0.865, 0.365, 0.401, 0.533, 0.405, 0.342, 0.332, 0.321, 0.318, 0.319, 0.

In [5]:
# Benchmark ML-KEM encapsulation
print(f"Benchmarking {alg} encapsulation...")
print("Warming up...")
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    for _ in range(100):
        ciphertext, shared_secret_enc = kem.encap_secret(public_key)
runs = 10000
encap_times_ms = []
encap_peaks_kb = []
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        ciphertext, shared_secret_enc = kem.encap_secret(public_key)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        encap_times_ms.append(elapsed_ms)
        encap_peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "encapsulation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean encapsulation time: {np.mean(encap_times_ms):.3f} ms")
print(f"Min encapsulation time: {np.min(encap_times_ms):.3f} ms")
print(f"Max encapsulation time: {np.max(encap_times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(encap_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(encap_peaks_kb):.3f} KB")
print("Encapsulation times (ms):", [round(t, 3) for t in encap_times_ms])
print("Peaks (KB):", [round(p, 3) for p in encap_peaks_kb])

append_to_output(
    benchmark_output_path,
    f"\nBenchmarking {alg} encapsulation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean encapsulation time: {np.mean(encap_times_ms):.3f} ms",
    f"Min encapsulation time: {np.min(encap_times_ms):.3f} ms",
    f"Max encapsulation time: {np.max(encap_times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(encap_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(encap_peaks_kb):.3f} KB")

Benchmarking ML-KEM-768 encapsulation...
Warming up...
Algorithm: ML-KEM-768
Runs: 10000
Mean encapsulation time: 0.437 ms
Min encapsulation time: 0.352 ms
Max encapsulation time: 5.017 ms
Mean peak memory: 3.760 KB
Max peak memory: 5.810 KB
Encapsulation times (ms): [0.394, 0.497, 0.967, 0.849, 0.556, 0.731, 0.615, 0.409, 0.418, 0.426, 1.391, 1.366, 1.374, 0.946, 1.037, 0.866, 0.915, 0.472, 0.693, 0.627, 0.661, 0.441, 0.686, 0.437, 0.555, 0.504, 0.399, 0.729, 0.587, 0.758, 0.398, 0.429, 0.618, 0.493, 0.745, 0.706, 1.761, 0.456, 0.444, 0.492, 0.388, 0.483, 0.592, 0.577, 0.618, 0.513, 0.399, 0.416, 0.464, 0.847, 0.783, 0.482, 0.401, 0.394, 0.381, 0.452, 0.737, 0.54, 0.436, 0.483, 0.442, 1.18, 0.738, 0.415, 0.522, 0.413, 0.482, 0.633, 0.403, 0.386, 0.453, 0.398, 0.44, 0.415, 0.704, 0.444, 0.662, 0.432, 0.487, 0.603, 0.43, 0.44, 0.451, 0.419, 0.403, 0.437, 0.405, 0.418, 0.4, 0.452, 0.466, 0.396, 0.405, 0.435, 0.563, 0.436, 0.422, 0.402, 0.623, 0.419, 0.457, 0.621, 0.603, 0.417, 0.398, 0.3

In [6]:
# Benchmark ML-KEM decapsulation
print(f"Benchmarking {alg} decapsulation...")
print("Warming up...")
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    ciphertext, shared_secret_enc = kem.encap_secret(public_key)
    for _ in range(100):
        shared_secret_dec = kem.decap_secret(ciphertext)
runs = 10000
decap_times_ms = []
decap_peaks_kb = []
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    ciphertext, shared_secret_enc = kem.encap_secret(public_key)
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        shared_secret_dec = kem.decap_secret(ciphertext)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        decap_times_ms.append(elapsed_ms)
        decap_peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "decapsulation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean decapsulation time: {np.mean(decap_times_ms):.3f} ms")
print(f"Min decapsulation time: {np.min(decap_times_ms):.3f} ms")
print(f"Max decapsulation time: {np.max(decap_times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(decap_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(decap_peaks_kb):.3f} KB")
print("Decapsulation times (ms):", [round(t, 3) for t in decap_times_ms])
print("Peaks (KB):", [round(p, 3) for p in decap_peaks_kb])

append_to_output(
    benchmark_output_path,
    f"\nBenchmarking {alg} decapsulation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean decapsulation time: {np.mean(decap_times_ms):.3f} ms",
    f"Min decapsulation time: {np.min(decap_times_ms):.3f} ms",
    f"Max decapsulation time: {np.max(decap_times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(decap_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(decap_peaks_kb):.3f} KB")

Benchmarking ML-KEM-768 decapsulation...
Warming up...
Algorithm: ML-KEM-768
Runs: 10000
Mean decapsulation time: 0.507 ms
Min decapsulation time: 0.362 ms
Max decapsulation time: 1.929 ms
Mean peak memory: 1.609 KB
Max peak memory: 1.609 KB
Decapsulation times (ms): [0.446, 0.497, 0.479, 0.502, 0.468, 0.448, 0.439, 0.472, 0.594, 0.717, 0.653, 0.835, 0.667, 0.509, 0.679, 0.535, 0.462, 0.434, 0.47, 0.432, 0.427, 0.427, 0.432, 0.425, 0.457, 0.48, 0.527, 0.479, 0.705, 0.614, 0.486, 0.687, 0.497, 0.462, 0.455, 0.452, 0.465, 0.46, 0.452, 0.47, 0.458, 0.521, 1.132, 0.541, 0.508, 0.479, 0.501, 0.606, 0.456, 0.524, 0.482, 0.471, 0.459, 0.674, 0.472, 0.509, 0.461, 0.455, 0.45, 0.779, 0.618, 0.607, 0.455, 0.458, 0.456, 0.453, 0.454, 0.439, 0.446, 0.456, 0.432, 0.432, 0.453, 0.442, 0.674, 0.471, 0.454, 0.849, 0.459, 0.467, 0.446, 0.46, 0.466, 0.565, 0.687, 0.492, 0.473, 0.454, 0.461, 0.474, 0.484, 0.471, 0.447, 0.448, 0.427, 0.452, 0.481, 0.463, 0.47, 0.586, 0.477, 0.466, 0.488, 0.602, 0.522, 0.4

In [7]:
# Measure ML-KEM size metrics
print(f"Measuring {alg} size metrics...")
with oqs.KeyEncapsulation(alg) as kem:
    public_key = kem.generate_keypair()
    secret_key = kem.export_secret_key()
    ciphertext, shared_secret = kem.encap_secret(public_key)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(public_key)} bytes")
print(f"Secret key size: {len(secret_key)} bytes")
print(f"Ciphertext size: {len(ciphertext)} bytes")
print(f"Shared secret size: {len(shared_secret)} bytes")

append_to_output(
    benchmark_output_path,
    f"\nMeasuring {alg} size metrics...",
    f"Algorithm: {alg}",
    f"Public key size: {len(public_key)} bytes",
    f"Secret key size: {len(secret_key)} bytes",
    f"Ciphertext size: {len(ciphertext)} bytes",
    f"Shared secret size: {len(shared_secret)} bytes")

Measuring ML-KEM-768 size metrics...
Algorithm: ML-KEM-768
Public key size: 1184 bytes
Secret key size: 2400 bytes
Ciphertext size: 1088 bytes
Shared secret size: 32 bytes


In [8]:
# ECDH-P-256 comparison benchmark
from cryptography.hazmat.primitives.asymmetric import ec
ecdh_runs = 10000

print("Benchmarking ECDH-P-256 keypair generation...")
ecdh_keygen_times = []
ecdh_keygen_peaks_kb = []
for run_index in range(ecdh_runs):
    start = time.perf_counter()
    tracemalloc.start()
    private_key = ec.generate_private_key(ec.SECP256R1())
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    ecdh_keygen_times.append(elapsed_ms)
    ecdh_keygen_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {ecdh_runs}")
print(f"Mean keypair time: {np.mean(ecdh_keygen_times):.3f} ms")
print(f"Min keypair time: {np.min(ecdh_keygen_times):.3f} ms")
print(f"Max keypair time: {np.max(ecdh_keygen_times):.3f} ms")
print(f"Mean peak memory: {np.mean(ecdh_keygen_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(ecdh_keygen_peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in ecdh_keygen_times])
print("Peaks (KB):", [round(p, 3) for p in ecdh_keygen_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking ECDH-P-256 keypair generation...",
    f"Runs: {ecdh_runs}",
    f"Mean keypair time: {np.mean(ecdh_keygen_times):.3f} ms",
    f"Min keypair time: {np.min(ecdh_keygen_times):.3f} ms",
    f"Max keypair time: {np.max(ecdh_keygen_times):.3f} ms",
    f"Mean peak memory: {np.mean(ecdh_keygen_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(ecdh_keygen_peaks_kb):.3f} KB")

print("Benchmarking ECDH-P-256 shared secret derivation...")
# Generate a fixed peer public key for consistent benchmarking
peer_private = ec.generate_private_key(ec.SECP256R1())
peer_public = peer_private.public_key()

ecdh_derive_times = []
ecdh_derive_peaks_kb = []
for run_index in range(ecdh_runs):
    # Generate new private key each time
    private_key = ec.generate_private_key(ec.SECP256R1())
    start = time.perf_counter()
    tracemalloc.start()
    shared_secret = private_key.exchange(ec.ECDH(), peer_public)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    ecdh_derive_times.append(elapsed_ms)
    ecdh_derive_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "shared_secret_derivation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {ecdh_runs}")
print(f"Mean derivation time: {np.mean(ecdh_derive_times):.3f} ms")
print(f"Min derivation time: {np.min(ecdh_derive_times):.3f} ms")
print(f"Max derivation time: {np.max(ecdh_derive_times):.3f} ms")
print(f"Mean peak memory: {np.mean(ecdh_derive_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(ecdh_derive_peaks_kb):.3f} KB")
print("Derivation times (ms):", [round(t, 3) for t in ecdh_derive_times])
print("Peaks (KB):", [round(p, 3) for p in ecdh_derive_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking ECDH-P-256 shared secret derivation...",
    f"Runs: {ecdh_runs}",
    f"Mean derivation time: {np.mean(ecdh_derive_times):.3f} ms",
    f"Min derivation time: {np.min(ecdh_derive_times):.3f} ms",
    f"Max derivation time: {np.max(ecdh_derive_times):.3f} ms",
    f"Mean peak memory: {np.mean(ecdh_derive_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(ecdh_derive_peaks_kb):.3f} KB")

print("Measuring ECDH-P-256 size metrics...")
# Use the last generated keys
private_bytes = private_key.private_bytes(
    encoding=Encoding.PEM,
    format=PrivateFormat.PKCS8,
    encryption_algorithm=NoEncryption(),
)
public_bytes = private_key.public_key().public_bytes(
    encoding=Encoding.PEM,
    format=PublicFormat.SubjectPublicKeyInfo,
)

print(f"Public key size: {len(public_bytes)} bytes")
print(f"Private key size: {len(private_bytes)} bytes")
print(f"Shared secret size: {len(shared_secret)} bytes")

append_to_output(
    rsa_benchmark_output_path,
    "\nMeasuring ECDH-P-256 size metrics...",
    f"Public key size: {len(public_bytes)} bytes",
    f"Private key size: {len(private_bytes)} bytes",
    f"Shared secret size: {len(shared_secret)} bytes")

Benchmarking ECDH-P-256 keypair generation...
Runs: 10000
Mean keypair time: 0.120 ms
Min keypair time: 0.071 ms
Max keypair time: 4.056 ms
Mean peak memory: 0.236 KB
Max peak memory: 1.060 KB
Times (ms): [4.056, 0.232, 0.231, 0.146, 0.099, 0.105, 0.197, 0.147, 0.213, 0.151, 0.185, 0.191, 0.101, 0.125, 0.197, 0.135, 0.108, 0.139, 0.288, 0.19, 0.142, 0.112, 0.214, 0.167, 0.161, 0.095, 0.201, 0.165, 0.094, 0.116, 0.085, 0.087, 0.095, 0.1, 0.124, 0.118, 0.192, 0.104, 0.117, 0.18, 0.151, 0.122, 0.099, 0.129, 0.166, 0.191, 0.138, 0.091, 0.083, 0.091, 0.098, 0.205, 0.173, 0.231, 0.291, 0.26, 0.205, 0.117, 0.086, 0.085, 0.115, 0.083, 0.081, 0.078, 0.128, 0.163, 0.16, 0.13, 0.108, 0.1, 0.09, 0.184, 0.128, 0.087, 0.098, 0.09, 0.236, 0.15, 0.129, 0.114, 0.088, 0.106, 0.132, 0.143, 0.087, 0.079, 0.078, 0.074, 0.071, 0.179, 0.192, 0.168, 0.22, 0.154, 0.117, 0.113, 0.107, 0.098, 0.169, 0.103, 0.094, 0.086, 0.127, 0.101, 0.105, 0.087, 0.083, 0.086, 0.085, 0.082, 0.085, 0.238, 0.126, 0.12, 0.09, 0.09